# Agentic RAG Prototípus

Ez a notebook egy LangGraph-ra épülő Agentic RAG architektúra működését mutatja be.

**Technológiai döntés (LLM):** A kezdetben tesztelt felhős szolgáltatások (Google Gemini Free Tier) API korlátozásai miatt a végleges prototípus a Hugging Face Inference API-t és a **Qwen 2.5 7B** nyílt forráskódú modellt alkalmaztam. A LangChain és a LangGraph absztrakcióinak köszönhetően ez az átállás bizonyítja a rendszer modularitását és autonóm döntéshozatali (routing) képességét.

In [33]:
import os
from dotenv import load_dotenv

load_dotenv()

# Most már a Hugging Face tokent ellenőrizzük, mert az a fő motor
if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    raise ValueError("Hiányzik a HUGGINGFACEHUB_API_TOKEN!")
else:
    print("Hugging Face API token sikeresen betöltve.")

Hugging Face API token sikeresen betöltve.


## 1. Tudásbázis felépítése 

Ebben a blokkban történik a RAG pipeline adatforrásának előkészítése. A script a beolvasott PDF-et kisebb, átfedéssel (overlap) rendelkező blokkokra darabolja, így elkerülve a fontos mondatok és kontextusok kettévágását.

**Technológiai döntés (Embedding):** Bár a szöveggenerálásért (LLM) egy felhős Hugging Face modell (Qwen 2.5) a felelős, a beágyazási (embedding) folyamatot egy lokális modellre (`all-MiniLM-L6-v2`) bíztam. Ennek a hibrid megközelítésnek az az oka, hogy a Cloud API-k ingyenes csomagjai szigorú kérés-korlátozással (rate limit) rendelkeznek, ami egy hosszabb dokumentum hálózaton keresztüli vektorizálásánál azonnal hibára futna. A gyors visszakeresést a rendszer egy memóriában futó FAISS vektoradatbázissal biztosítja.

In [34]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

file_path = "data/ZsigoB_irodalomkutatas.pdf"
try:
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    print(f"Betöltve {len(docs)} oldal a dokumentumból.")
except Exception as e:
    print(f"Hiba a fájl beolvasásakor: {e}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200 
)
splits = text_splitter.split_documents(docs)
print(f"A dokumentum {len(splits)} darabra (chunk) lett felosztva.")

print("Lokális HuggingFace beágyazó modell betöltése (ez eltarthat pár másodpercig)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vektoradatbázis és retriever sikeresen létrehozva és memóriába töltve!")

Betöltve 45 oldal a dokumentumból.
A dokumentum 130 darabra (chunk) lett felosztva.
Lokális HuggingFace beágyazó modell betöltése (ez eltarthat pár másodpercig)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vektoradatbázis és retriever sikeresen létrehozva és memóriába töltve!


## 2. Agentic Workflow implementálása

A projekt célja, hogy a chatbot autonóm döntéshozatalt és részfeladat-végrehajtást demonstráljon. A megvalósítás az előírt **LangGraph** keretrendszerre épül, amellyel egy jól strukturált állapotgépet (state machine) alakítottam ki.

**Architekturális felépítés:**
1. **State (Állapot):** Az ágens működése során egy közös szótárban (`AgentState`) tárolja az aktuális kérdést, a vektoradatbázisból kinyert kontextust és a végső választ.
2. **Retrieve Node:** Felelős a korábban inicializált FAISS adatbázisból a releváns dokumentum-részletek lekérdezéséért.
3. **Generate Node:** A Hugging Face felhőjében futó **Qwen 2.5 7B** modell felhasználásával, a kinyert kontextus alapján fogalmazza meg a választ. A hallucináció esélyét egy szigorú RAG-specifikus prompttal minimalizáltam.

In [35]:
import os
from typing import TypedDict, List
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    question: str
    context: List[str]
    answer: str

endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1, 
    huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN")
)

llm = ChatHuggingFace(llm=endpoint)

def retrieve_node(state: AgentState):
    print("ADATOK VISSZAKERESÉSE")
    question = state["question"]
    
    documents = retriever.invoke(question)
    context = [doc.page_content for doc in documents]
    
    return {"context": context}

def generate_node(state: AgentState):
    print("VÁLASZ GENERÁLÁSA")
    question = state["question"]
    context_str = "\n\n".join(state["context"])
    
    prompt = f"Kérlek, kizárólag a következő kontextus alapján válaszolj a kérdésre magyar nyelven! Ha nincs benne a válasz, írd azt: 'Nincs erről információm.'\n\nKontextus: {context_str}\n\nKérdés: {question}"
    
    response = llm.invoke(prompt)
    return {"answer": response.content}

In [36]:
workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
print("Agentic RAG LangGraph sikeresen összeállítva és lefordítva.")

Agentic RAG LangGraph sikeresen összeállítva és lefordítva.


In [38]:
test_question = "Mi a dokumentum központi témája?"

print(f"Kérdés: {test_question}\n")

inputs = {"question": test_question}
for output in app.stream(inputs):
    for key, value in output.items():
        print(f"Csomópont '{key}' sikeresen lefutott.")

print("\nVégső válasz:")
print(output["generate"]["answer"])

Kérdés: Mi a dokumentum központi témája?

ADATOK VISSZAKERESÉSE
Csomópont 'retrieve' sikeresen lefutott.
VÁLASZ GENERÁLÁSA
Csomópont 'generate' sikeresen lefutott.

Végső válasz:
A dokumentum központi témája a Q-függvény és a politika iteratív finomításával kapcsolatos tanulási folyamat a modern RL (Reinforcement Learning) algoritmusokban.


## 3. Teljesítmény, Szűk keresztmetszetek (Bottlenecks) és Tesztelés

A prototípus fejlesztése és tesztelése során az alábbi architekturális kihívásokat és szűk keresztmetszeteket azonosítottam:

### 1. Szűk keresztmetszetek (Bottlenecks) és Megoldásuk
* **API Rate Limiting és Instabilitás (Kritikus hiba):** A legfőbb szűk keresztmetszetet a felhő alapú ingyenes API-k jelentették. A Google Gemini Free Tier használata azonnal `429 Resource Exhausted` hibát dobott a szigorú korlátok miatt. A Hugging Face Inference API esetében pedig szerveroldali átirányítási (routing) hibák léptek fel (nem támogatott taskok harmadik félnél).
* **Az Architektúra Rugalmassága:** A fenti bottleneckek kiküszöbölésére a rendszert a LangChain `ChatHuggingFace` wrapperével stabilizáltam, és a **Qwen 2.5 7B** modellt kötöttem be. A LangGraph pipeline könnyen cserélhető komponensekre épül, így egy szolgáltatói hiba esetén a modell pillanatok alatt leváltható.

### 2. A Teljesítmény Mérési és Tesztelési Stratégiája
Egy éles (produkciós) rendszerben a manuális tesztelés helyett az alábbi automatizált metrikákat alkalmaznám:
* **Válaszok Minősége (RAGAS):** A Retrieval Augmented Generation Assessment (RAGAS) keretrendszerrel mérném a kimenetet. Külön pontoznám a *Faithfulness* (a modell csak a kontextusból válaszol-e) és az *Answer Relevance* (mennyire releváns a válasz a kérdésre) metrikákat.
* **Terhelésteszt (Load Testing):** Locust vagy k6 eszközökkel szimulálnám a párhuzamos lekérdezéseket, hogy mérjem a FAISS adatbázis és a felhős API válaszidejét.

### 3. Továbbfejlesztési Javaslatok
1. **Advanced Agentic Routing:** Új csomópontok beépítése a LangGraph-ba, pl. egy *Self-Reflective* node, ami ellenőrzi, hogy a FAISS visszadott-e releváns adatot, és ha nem, módosítja a keresési kulcsszavakat (Query Rewriting).
2. **Költséghatékony Lokális LLM:** A felhős API-k teljes kiváltása egy dedikált szerverre telepített `vLLM` példánnyal, ami jelentősen csökkenti a hálózati késleltetést.